In [1]:
import glob
import json
import os
import random
import uuid
from itertools import product

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.signal import stft, welch
from scipy.stats import entropy, norm

# Scikit-Learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras & PyTorch
import tensorflow as tf
from tensorflow.keras import Model, callbacks, layers, models
import torch

# ---------------------------------------------------------------------------
# REPRODUCIBILITY & HARDWARE SETUP
# ---------------------------------------------------------------------------
print("Script execution started...")
SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# Set seeds for Python, NumPy, TensorFlow, and PyTorch
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

print(f"Reproducibility settings locked with SEED: {SEED}")

# ---------------------------------------------------------------------------
# HARDWARE ACCELERATION CHECK
# ---------------------------------------------------------------------------
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

Script execution started...
Reproducibility settings locked with SEED: 42
No GPU detected for TensorFlow. Using CPU.


2026-08-28 05:32:28.808181: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
# Path to the participants TSV file
file_path = "/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /participants.tsv"

# Read the tab-separated file
df = pd.read_csv(file_path, sep='\t')

# Display the first few rows
df.head()

,participant_id,subject_id,group,updrs_part_iii,updrs_total,moca,age,sex,disease_duration,ledd,pigd_score,td_score,ctt
0,sub-001,HC0001,HC,0.0,0.0,30.0,42.0,M,NaN,NaN,NaN,NaN,NaN
1,sub-002,HC0003,HC,2.0,3.0,27.0,60.0,M,NaN,NaN,NaN,NaN,66.0
2,sub-003,HC0004,HC,0.0,1.0,27.0,60.0,F,NaN,NaN,NaN,NaN,63.0
3,sub-004,HC0005,HC,1.0,1.0,25.0,72.0,M,NaN,NaN,NaN,NaN,116.0
4,sub-005,HC0006,HC,NaN,NaN,NaN,47.0,M,NaN,NaN,NaN,NaN,NaN


In [3]:
sub_condition = df.iloc[:,2].values
print(sub_condition[0:5])

['HC' 'HC' 'HC' 'HC' 'HC']


In [4]:
print(df.shape)

(144, 13)


In [5]:
missing_ids = []

for i in range(1, 145):
    # Format i with 3-digit zero-padding (e.g., 001, 002, ..., 144)
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-walk_eeg.set"
    
    # Check if the file does NOT exist and print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

1
5
16
20
25
36
43
84
100
120
126


In [6]:


# Format participant ID as 3-digit zero-padded string ('001')
sub_id = f"{1:03d}"
file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

# Load the file into memory
raw = mne.io.read_raw_eeglab(file_path, preload=True)

# Extract raw numerical array: shape is (Channels, Length)
signal = raw.get_data()

print("Signal shape (C, L):", signal.shape)

Signal shape (C, L): (65, 60964)


/tmp/ipykernel_58/1231990765.py:6: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True)


In [7]:
sfreq = raw.info['sfreq']

print(f"Sampling Frequency: {sfreq} Hz")

Sampling Frequency: 250.0 Hz


In [8]:
def get_eeg_signal(
    sub_id, 
    task="walk", 
    band="full",
    target_sfreq=256,
    duration=2.0, 
    notch_freq=50.0, 
    reject_threshold=0.00028,
    temporal_only=True,
    base_dir="/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 "
):
    """
    Loads, cleans, filters by frequency band, selects 10 temporal channels, 
    resamples, and segments EEG data.
    Returns float32 NumPy array of shape (N_epochs, Channels, Time).
    """
    # 1. Map band names to frequency limits
    band_limits = {
        'full':  (1.0, 45.0),
        'delta': (1.0, 4.0),
        'theta': (4.0, 8.0),
        'alpha': (8.0, 12.0),
        'beta':  (12.0, 30.0),
        'gamma': (30.0, 45.0)
    }
    
    if band.lower() not in band_limits:
        raise ValueError(f"Invalid band '{band}'. Choose from: {list(band_limits.keys())}")
        
    l_freq, h_freq = band_limits[band.lower()]

    # 2. Format subject ID
    if isinstance(sub_id, int):
        sub_str = f"{sub_id:03d}"
    else:
        sub_str = str(sub_id).zfill(3)

    file_path = f"{base_dir}/sub-{sub_str}/eeg/sub-{sub_str}_task-{task}_eeg.set"

    # 3. Load continuous file
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

    # 4. Fix channel types
    channel_type_mapping = {
        'EOG1': 'eog', 'EOG2': 'eog', 'EOG3': 'eog', 'EOG4': 'eog', 'VREF': 'misc'
    }
    existing_mapping = {ch: t for ch, t in channel_type_mapping.items() if ch in raw.ch_names}
    if existing_mapping:
        raw.set_channel_types(existing_mapping)

    # 5. ISOLATE TEMPORAL, FRONTO-TEMPORAL & PARIETO-TEMPORAL CHANNELS
    if temporal_only:
        # 10 Temporal-region channels present in your 65-channel dataset layout
        target_temporal_patterns = [
            'T7', 'T8', 'T9', 'T10',
            'FT7', 'FT8',
            'TP7', 'TP8', 'TP9', 'TP10'
        ]
        
        # Match case-insensitive channel names existing in raw file
        available_channels = [
            ch for ch in raw.ch_names 
            if any(ch.upper() == pattern.upper() for pattern in target_temporal_patterns)
        ]
        
        if len(available_channels) > 0:
            raw.pick_channels(available_channels)
        else:
            # Fallback to general EEG picking if exact channel names differ
            raw.pick_types(eeg=True, eog=False, misc=False)
    else:
        raw.pick_types(eeg=True, eog=False, misc=False)

    # 6. PREPROCESSING
    # A. Bandpass filter for selected band
    raw.filter(l_freq=l_freq, h_freq=h_freq, fir_design='firwin', verbose=False)

    # B. Notch Filter
    if notch_freq is not None and h_freq >= notch_freq:
        raw.notch_filter(freqs=notch_freq, verbose=False)

    # C. Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average', projection=False, verbose=False)

    # 7. Resample to target frequency
    raw.resample(sfreq=target_sfreq, verbose=False)

    # 8. Epoching with Artifact Rejection
    events = mne.make_fixed_length_events(raw, duration=duration)
    reject_criteria = dict(eeg=reject_threshold) if reject_threshold is not None else None

    epochs = mne.Epochs(
        raw, 
        events=events, 
        tmin=0, 
        tmax=duration - (1 / target_sfreq), 
        baseline=None, 
        reject=reject_criteria,
        preload=True, 
        verbose=False
    )

    # Extract array and cast to float32 to save RAM
    signal = epochs.get_data().astype(np.float32)

    return signal

In [9]:
non_walk_ids = [1,5,16,20,25,36,43,84,100,120,126]

In [10]:
rest_ids = [i for i in range(1,145)]
walk_ids = [i for i in range(1,145) if i not in non_walk_ids]

In [11]:
def get_data(task, band, rest_ids, walk_ids):
    X_hc = []
    X_pd = []
    
    # Select subject list based on task
    sub_ids = rest_ids if task == "rest" else walk_ids
    
    for sub_id in sub_ids:
        try:
            # Extract signal for the current subject
            eeg_signal = get_eeg_signal(sub_id=sub_id, task=task, band=band)
            
            # Split into HC (< 29) or PD (>= 29)
            if sub_id < 29:
                X_hc.append(eeg_signal)
            else:
                X_pd.append(eeg_signal)
                
        except Exception as e:
            print(f"Skipping Subject {sub_id} ({task}, {band}) due to error: {e}")
            
    return X_hc, X_pd


In [12]:
# --- Example Usage ---
# Extract Beta band for walking task at 128 Hz
beta_walk = get_eeg_signal(sub_id=3, task="walk", band="beta")

# Extract Full spectrum (1-45 Hz) for resting task at 128 Hz
full_rest = get_eeg_signal(sub_id=3, task="rest", band="full")

print("Beta Walk Signal shape (N, C, T):", beta_walk.shape)  # e.g., (N, 60, 256)
print("Full Rest Signal shape (N, C, T):", full_rest.shape)  # e.g., (N, 60, 256)

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/549130017.py:41: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/549130017.py:49: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Beta Walk Signal shape (N, C, T): (121, 10, 512)
Full Rest Signal shape (N, C, T): (121, 10, 512)


/tmp/ipykernel_58/549130017.py:41: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/549130017.py:49: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


In [13]:
non_walk_ids = [1,5,16,20,25,36,43,84,100,120,126]

In [14]:
rest_ids = [i for i in range(1,145)]
walk_ids = [i for i in range(1,145) if i not in non_walk_ids]

In [15]:
def get_data(task, band, rest_ids, walk_ids):
    X_hc = []
    X_pd = []
    
    # Select subject list based on task
    sub_ids = rest_ids if task == "rest" else walk_ids
    
    for sub_id in sub_ids:
        try:
            # Extract signal for the current subject
            eeg_signal = get_eeg_signal(sub_id=sub_id, task=task, band=band)
            
            # Split into HC (< 29) or PD (>= 29)
            if sub_id < 29:
                X_hc.append(eeg_signal)
            else:
                X_pd.append(eeg_signal)
                
        except Exception as e:
            print(f"Skipping Subject {sub_id} ({task}, {band}) due to error: {e}")
            
    return X_hc, X_pd


In [16]:
# --- Example Usage ---
# Extract Beta band for walking task at 128 Hz
beta_walk = get_eeg_signal(sub_id=3, task="walk", band="beta")

# Extract Full spectrum (1-45 Hz) for resting task at 128 Hz
full_rest = get_eeg_signal(sub_id=3, task="rest", band="full")

print("Beta Walk Signal shape (N, C, T):", beta_walk.shape)  # e.g., (N, 60, 256)
print("Full Rest Signal shape (N, C, T):", full_rest.shape)  # e.g., (N, 60, 256)

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/549130017.py:41: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/549130017.py:49: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)
/tmp/ipykernel_58/549130017.py:41: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/549130017.py:49: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


Beta Walk Signal shape (N, C, T): (121, 10, 512)
Full Rest Signal shape (N, C, T): (121, 10, 512)


In [17]:
non_walk_ids = [1,5,16,20,25,36,43,84,100,120,126]

In [18]:
rest_ids = [i for i in range(1,145)]
walk_ids = [i for i in range(1,145) if i not in non_walk_ids]

In [19]:
def get_data(task, band, rest_ids, walk_ids):
    X_hc = []
    X_pd = []
    
    # Select subject list based on task
    sub_ids = rest_ids if task == "rest" else walk_ids
    
    for sub_id in sub_ids:
        try:
            # Extract signal for the current subject
            eeg_signal = get_eeg_signal(sub_id=sub_id, task=task, band=band)
            
            # Split into HC (< 29) or PD (>= 29)
            if sub_id < 29:
                X_hc.append(eeg_signal)
            else:
                X_pd.append(eeg_signal)
                
        except Exception as e:
            print(f"Skipping Subject {sub_id} ({task}, {band}) due to error: {e}")
            
    return X_hc, X_pd


In [20]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1):
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]
    
    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)
    
    if total_c0 == total_c1:
        return np.concatenate(X_list_c0, axis=0), np.concatenate(X_list_c1, axis=0)

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active
        
        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break
            
        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0
                
                available = maj_counts[i] - allocations[i]
                take = min(share, available)
                
                allocations[i] += take
                remaining_target -= take
                
                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])
            
    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [21]:
def scale_data(X_list):
    scaled = []
    for sub in X_list:
        flat = sub.reshape(-1, sub.shape[-1])
        mu = np.mean(flat, axis=0)
        std = np.std(flat, axis=0) + 1e-8
        scaled.append((sub - mu) / std)
    return scaled

In [22]:


class PureSpectralConv1D(layers.Layer):
    """
    Pure 1D Fourier Integral Operator:
    Computes (K(a) * v)(t) = F^-1( R_phi * F(v) )(t) continuously across domain resolution.
    """
    def __init__(self, in_channels, out_channels, modes1, **kwargs):
        super(PureSpectralConv1D, self).__init__(**kwargs)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.scale = 1.0 / (in_channels * out_channels)

    def build(self, input_shape):
        self.weights1_real = self.add_weight(
            shape=(self.in_channels, self.out_channels, self.modes1),
            initializer=tf.random_normal_initializer(stddev=self.scale),
            trainable=True, name="w_real"
        )
        self.weights1_imag = self.add_weight(
            shape=(self.in_channels, self.out_channels, self.modes1),
            initializer=tf.random_normal_initializer(stddev=self.scale),
            trainable=True, name="w_imag"
        )
        super(PureSpectralConv1D, self).build(input_shape)

    def call(self, x):
        time_steps = tf.shape(x)[1]
        
        # 1. Continuous Fourier Transform
        x_transposed = tf.transpose(x, perm=[0, 2, 1])
        x_ft = tf.signal.rfft(x_transposed)
        
        # 2. Spectral Kernel Multiplication R_phi (Lower modes operator)
        x_ft_sub = x_ft[:, :, :self.modes1]
        weights1 = tf.complex(self.weights1_real, self.weights1_imag)
        out_ft = tf.einsum("bix,iox->box", x_ft_sub, weights1)
        
        # 3. Dynamic Zero-Padding for exact resolution recovery
        pad_len = (time_steps // 2 + 1) - self.modes1
        paddings = tf.stack([
            tf.constant([0, 0]), 
            tf.constant([0, 0]), 
            tf.stack([0, pad_len])
        ])
        out_ft_padded = tf.pad(out_ft, paddings)
        
        # 4. Inverse Fourier Transform back to temporal function space
        x_out = tf.signal.irfft(out_ft_padded, fft_length=[time_steps])
        return tf.transpose(x_out, perm=[0, 2, 1])


class PureFNO1D(Model):
    """
    Pure 1D Fourier Neural Operator (FNO) Architecture:
    Function-to-Function Mapping Framework G: A -> U
    
    Structure:
    1. Lifting Operator (P): Maps channel space to high-dim continuous space
    2. Stacked Fourier Layers (K_i + W_i): Operates strictly in function space
    3. Projection Operator (Q): Maps latent representation to continuous output field u(t)
    4. Domain Integration: Approximates continuous integral int_Omega u(t) dt
    """
    def __init__(self, in_channels=16, modes=12, width=32, **kwargs):
        super(PureFNO1D, self).__init__(**kwargs)
        
        # Permute (N, C, T) -> (N, T, C)
        self.permute = layers.Permute((2, 1))
        
        # 1. LIFTING OPERATOR P: a(t) -> v_0(t)
        self.p_lifting = layers.Dense(width)
        
        # 2. FOURIER OPERATOR LAYER 1: v_0(t) -> v_1(t)
        self.fno1 = PureSpectralConv1D(in_channels=width, out_channels=width, modes1=modes)
        self.w1 = layers.Conv1D(width, kernel_size=1)  # Local spatial linear transformation W
        
        # FOURIER OPERATOR LAYER 2: v_1(t) -> v_2(t)
        self.fno2 = PureSpectralConv1D(in_channels=width, out_channels=width, modes1=modes)
        self.w2 = layers.Conv1D(width, kernel_size=1)
        
        # 3. PROJECTION OPERATOR Q: v_2(t) -> u(t) (Target Continuous Field)
        self.q_proj1 = layers.Dense(128, activation='gelu')
        self.q_proj2 = layers.Dense(1)  # Evaluates output continuous field u(t)
        
    def call(self, inputs):
        # inputs shape: (N, Channels, Time)
        x = self.permute(inputs)  # -> (N, Time, Channels)
        
        # Step 1: Lift to higher-dimensional continuous space
        x = self.p_lifting(x)
        
        # Step 2: Pass through Non-linear Fourier Integral Operators
        # Layer 1: v_1 = activation( K_1(v_0) + W_1(v_0) )
        x = tf.nn.gelu(self.fno1(x) + self.w1(x))
        
        # Layer 2: v_2 = activation( K_2(v_1) + W_2(v_1) )
        x = tf.nn.gelu(self.fno2(x) + self.w2(x))
        
        # Step 3: Project back to target scalar field u(t) across domain
        x = self.q_proj1(x)
        u_t = self.q_proj2(x)  # Shape: (N, Time, 1) - Continuous functional field
        
        # Step 4: Domain Integration (Continuous Operator Functional Evaluation)
        # Numerical approximation of integral over domain: Y = Sigmoid( 1/T * int_0^T u(t) dt )
        domain_integral = tf.reduce_mean(u_t, axis=1)  # Shape: (N, 1)
        return tf.nn.sigmoid(domain_integral)

In [23]:

def run_subject_level_mc_cv_optimized(X_healthy, X_pd, SEED=42):
    X_healthy = scale_data(X_healthy)
    X_pd = scale_data(X_pd)
    
    # Automatically infer channel count from input arrays (shape: N_epochs, Channels, Time)
    n_channels = X_healthy[0].shape[1]
    
    n_hc, n_pd = len(X_healthy), len(X_pd)
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    thresholds = list(range(65, 95, 5))
    
    hc_splits = list(outer_kf.split(np.arange(n_hc)))
    pd_splits = list(outer_kf.split(np.arange(n_pd)))
    
    total_correct = 0
    total_subjects = 0
    fold_summary_records = []
    
    # Pure FNO Hyperparameter Grid Search (Function-to-Function Operator Space)
    param_grid = {
        'lr': [1e-3],
        'batch_size': [32],
        'modes': [12],
        'width': [32]
    }
    
    keys = param_grid.keys()
    all_combinations = [dict(zip(keys, combo)) for combo in product(*param_grid.values())]
    
    for fold in range(5):
        print(f"\n========================================")
        print(f"========== OUTER FOLD {fold+1} / 5 ==========")
        print(f"========================================")
        
        hc_train_all, hc_test = hc_splits[fold]
        pd_train_all, pd_test = pd_splits[fold]
        
        best_score = -1.0
        best_params = None
        best_threshold = 75
        
        hc_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(hc_train_all))
        pd_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(pd_train_all))
        
        for params in all_combinations:
            inner_fold_accuracies = []
            inner_fold_thresholds = []
            
            for inner_fold in range(3):
                hc_tr_in_idx, hc_val_in_idx = hc_inner_splits[inner_fold]
                pd_tr_in_idx, pd_val_in_idx = pd_inner_splits[inner_fold]
                
                hc_train_sub = [X_healthy[hc_train_all[i]] for i in hc_tr_in_idx]
                pd_train_sub = [X_pd[pd_train_all[i]] for i in pd_tr_in_idx]
                hc_val_sub = [X_healthy[hc_train_all[i]] for i in hc_val_in_idx]
                pd_val_sub = [X_pd[pd_train_all[i]] for i in pd_val_in_idx]
                
                # Balance classes for inner training
                X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_sub, pd_train_sub)
                X_inner_train = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
                y_inner_train = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)
                
                # Shuffle training data
                shuffle_idx = np.random.RandomState(SEED).permutation(len(X_inner_train))
                X_inner_train = X_inner_train[shuffle_idx]
                y_inner_train = y_inner_train[shuffle_idx]
                
                # Explicit Train/Val split
                val_size = int(len(X_inner_train) * 0.1)
                X_tr, y_tr = X_inner_train[val_size:], y_inner_train[val_size:]
                X_va, y_va = X_inner_train[:val_size], y_inner_train[:val_size]

                # Instantiate Pure FNO Model (Continuous Operator Mapping)
                inner_model = PureFNO1D(
                    in_channels=n_channels,
                    modes=params['modes'], 
                    width=params['width']
                )
                inner_model.compile(
                    optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr']), 
                    loss='binary_crossentropy',
                    metrics=['accuracy']
                )
                
                early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
                inner_model.fit(
                    X_tr, y_tr, 
                    epochs=40, batch_size=params['batch_size'], 
                    verbose=0, validation_data=(X_va, y_va), callbacks=[early_stop]
                )
                
                # Inner validation threshold tuning
                val_subjects = hc_val_sub + pd_val_sub
                val_labels = [0]*len(hc_val_sub) + [1]*len(pd_val_sub)
                
                val_subject_ratios = []
                valid_val_labels = []
                
                for sub, true_lbl in zip(val_subjects, val_labels):
                    sub_array = np.asarray(sub, dtype=np.float32)
                    
                    if sub_array.ndim == 2:
                        sub_array = np.expand_dims(sub_array, axis=0)
                        
                    if sub_array.shape[0] == 0:
                        continue
                        
                    epoch_probs = inner_model.predict(sub_array, batch_size=params['batch_size'], verbose=0).flatten()
                    pct_pd = float(np.mean(epoch_probs) * 100)
                    val_subject_ratios.append(pct_pd)
                    valid_val_labels.append(true_lbl)

                best_t_inner, max_inner_acc = 75, -1.0
                for t in thresholds:
                    t_preds = [1 if ratio >= t else 0 for ratio in val_subject_ratios]
                    acc = accuracy_score(valid_val_labels, t_preds) if len(valid_val_labels) > 0 else 0.0
                    if acc > max_inner_acc:
                        max_inner_acc = acc
                        best_t_inner = t
                
                inner_fold_accuracies.append(max_inner_acc)
                inner_fold_thresholds.append(best_t_inner)
            
            mean_inner_acc = np.mean(inner_fold_accuracies)
            if mean_inner_acc > best_score:
                best_score = mean_inner_acc
                best_params = params
                # Take median of inner thresholds to preserve steps of 5
                best_threshold = int(np.median(inner_fold_thresholds))
        
        print(f">> Best Grid Parameters Selected: {best_params} | Threshold: {best_threshold}% (Inner Acc: {best_score:.4f})")
        
        # --- OUTER TRAINING & TESTING ---
        hc_train_final = [X_healthy[i] for i in hc_train_all]
        pd_train_final = [X_pd[i] for i in pd_train_all]
        
        X_tr_hc_final, X_tr_pd_final = balance_matrices_subject_wise(hc_train_final, pd_train_final)
        X_train_final = np.concatenate([X_tr_hc_final, X_tr_pd_final], axis=0)
        y_train_final = np.concatenate([np.zeros(len(X_tr_hc_final)), np.ones(len(X_tr_pd_final))], axis=0)
        
        shuffle_idx_final = np.random.RandomState(SEED).permutation(len(X_train_final))
        X_train_final = X_train_final[shuffle_idx_final]
        y_train_final = y_train_final[shuffle_idx_final]
        
        val_size_final = int(len(X_train_final) * 0.1)
        X_tr_f, y_tr_f = X_train_final[val_size_final:], y_train_final[val_size_final:]
        X_va_f, y_va_f = X_train_final[:val_size_final], y_train_final[:val_size_final]

        final_model = PureFNO1D(
            in_channels=n_channels,
            modes=best_params['modes'], 
            width=best_params['width']
        )
        final_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['lr']), 
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        
        early_stop_final = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        final_model.fit(
            X_tr_f, y_tr_f, 
            epochs=80, batch_size=best_params['batch_size'], 
            verbose=0, validation_data=(X_va_f, y_va_f), callbacks=[early_stop_final]
        )
        
        test_subjects = [X_healthy[i] for i in hc_test] + [X_pd[i] for i in pd_test]
        test_labels = [0]*len(hc_test) + [1]*len(pd_test)
        n_hc_test = len(hc_test)
        n_pd_test = len(pd_test)
        
        hc_correct_count = 0
        pd_correct_count = 0
        
        for sub, true_label in zip(test_subjects, test_labels):
            sub_array = np.asarray(sub, dtype=np.float32)
            if sub_array.ndim == 2:
                sub_array = np.expand_dims(sub_array, axis=0)
                
            if sub_array.shape[0] == 0:
                continue
                
            pct_pd = float(np.mean(final_model.predict(sub_array, batch_size=best_params['batch_size'], verbose=0).flatten()) * 100)
            
            vote_thresholds = [best_threshold - 5, best_threshold, best_threshold + 5]
            votes = [1 if pct_pd >= t else 0 for t in vote_thresholds]
            pred = 1 if sum(votes) >= 2 else 0
            
            if pred == true_label:
                if true_label == 0:
                    hc_correct_count += 1
                else:
                    pd_correct_count += 1
                    
        fold_total_correct = hc_correct_count + pd_correct_count
        fold_total_subjects = len(test_subjects)
        
        total_correct += fold_total_correct
        total_subjects += fold_total_subjects
        
        fold_summary_records.append({
            'Fold Number': fold + 1,
            'Optimal Hyperparams': str(best_params),
            'Healthy Correct': f"{hc_correct_count}/{n_hc_test}",
            'PD Correct': f"{pd_correct_count}/{n_pd_test}",
            'Total Correct': f"{fold_total_correct}/{fold_total_subjects}"
        })
        
        print(f"Outer Fold {fold+1} Stats -> Healthy: {hc_correct_count}/{n_hc_test} | PD: {pd_correct_count}/{n_pd_test} | Total: {fold_total_correct}/{fold_total_subjects}")

    summary_df = pd.DataFrame(fold_summary_records)
    print(f"\n========================================")
    print(f"Total Combined Correct: {total_correct}/{total_subjects}")
    print("\n--- Nested Cross-Validation Summary ---")
    print(summary_df.to_string(index=False))
    
    return summary_df

In [24]:
task = 'rest'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)
print(df)

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/549130017.py:41: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/549130017.py:49: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/549130017.py:41: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/549130017.py:49: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)
/tmp/ipykernel_58/549130017.py:41: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/549130017.py:49: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


...

Outer Fold 4 Stats -> Healthy: 5/5 | PD: 16/23 | Total: 21/28

========== OUTER FOLD 5 / 5 ==========


2026-08-28 06:28:45.862831: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-28 06:30:23.834026: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32} | Threshold: 65% (Inner Acc: 0.6977)


2026-08-28 06:34:54.917930: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-28 06:38:23.355135: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 3/5 | PD: 16/23 | Total: 19/28

Total Combined Correct: 90/144

--- Nested Cross-Validation Summary ---
 Fold Number                                       Optimal Hyperparams Healthy Correct PD Correct Total Correct
           1 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             5/6      11/24         16/30
           2 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             4/6      11/23         15/29
           3 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             5/6      14/23         19/29
           4 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             5/5      16/23         21/28
           5 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             3/5      16/23         19/28
   Fold Number                                Optimal Hyperparams  \
0            1  {'lr': 0.001, 'batch_size': 32, 'modes': 12, '...   
1            2  {'lr': 0.001, 'batch_size': 32, 'modes'

In [25]:
task = 'walk'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)
print(df)

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/549130017.py:41: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/549130017.py:49: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)
/tmp/ipykernel_58/549130017.py:41: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/549130017.py:49: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_58/549130017.py:41: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_58/549130017.py:49: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


...

Outer Fold 4 Stats -> Healthy: 3/4 | PD: 9/22 | Total: 12/26

========== OUTER FOLD 5 / 5 ==========


2026-08-28 07:04:48.231197: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-28 07:06:12.128115: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32} | Threshold: 65% (Inner Acc: 0.6844)


2026-08-28 07:08:52.845833: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-28 07:11:06.071674: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 1/4 | PD: 8/22 | Total: 9/26

Total Combined Correct: 64/133

--- Nested Cross-Validation Summary ---
 Fold Number                                       Optimal Hyperparams Healthy Correct PD Correct Total Correct
           1 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             4/5      10/22         14/27
           2 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             2/5      11/22         13/27
           3 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             5/5      11/22         16/27
           4 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             3/4       9/22         12/26
           5 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             1/4       8/22          9/26
   Fold Number                                Optimal Hyperparams  \
0            1  {'lr': 0.001, 'batch_size': 32, 'modes': 12, '...   
1            2  {'lr': 0.001, 'batch_size': 32, 'modes': 